# tcOCR — chạy thử trên Google Colab

OCR tài chính chứng khoán 3 lớp (pluggable) + Gradio UI.

**Trước khi chạy:** Runtime → Change runtime type → chọn **GPU (T4)** (cho VietOCR/Qwen; Paddle chạy CPU vẫn ổn khi test ít trang).

⚠️ Link Gradio `share=True` là công khai — chỉ test tài liệu **báo cáo tài chính công khai**, KHÔNG dùng data thật của khách.

## 1. Lấy code

In [ ]:
import os
if not os.path.exists('tcocr/tcocr'):
    !git clone -b claude/ocr-finance-99-percent-b5o5rh https://github.com/vnkiddev/tcocr.git
%cd tcocr
!git pull   # đảm bảo lấy bản mới nhất

## 2. Cài thư viện (~5-8 phút lần đầu)

Dùng **paddlepaddle CPU** (bản `-gpu` trên PyPI hay lỗi build trên Colab). Test ít trang thì CPU vẫn nhanh đủ dùng.

In [ ]:
!pip install -q numpy opencv-python-headless pillow PyMuPDF pdfplumber gradio
!pip uninstall -y -q paddlepaddle-gpu paddlepaddle 2>/dev/null
!pip install -q paddlepaddle paddleocr
!pip install -q vietocr
!pip install -q 'transformers>=4.49' accelerate qwen-vl-utils bitsandbytes sentencepiece
print('Cai xong.')
print('>>> BAT BUOC: Runtime -> Restart session (Ctrl+M .) MOT LAN, roi chay tiep tu cell 3. <<<')
print('    Paddle co extension bien dich, khong restart thi import se bao thieu du da cai.')

## 3. Kiểm tra import (chạy SAU khi đã Restart session)

Nếu cell này in ra version là ổn. Nếu lỗi → xem thông báo, thường chỉ cần Restart thêm lần nữa.

In [ ]:
%cd /content/tcocr
import paddle, paddleocr
print('paddle    :', paddle.__version__)
print('paddleocr :', paddleocr.__version__)
from tcocr.pipeline import OCRPipeline
print('tcocr import OK')

## 4. Khởi động Gradio UI

Mặc định: OCR=Paddle, escalation=null (tắt VLM cho nhẹ), correction=protonx.
Trên UI đổi backend để so sánh. Bật `local_vlm` khi muốn test Qwen2.5-VL (cần GPU).

**Nếu "không thấy app lên":**
1. Cell **chạy mãi không dừng là ĐÚNG** (giữ server sống) — đừng đợi nó "xong".
2. UI hiện **ngay dưới cell** (inline) — cuộn xuống xem trước.
3. Link `*.gradio.live` lần đầu mở đợi **30–60s**; trang trắng → đã fix bằng `ssr_mode=False` (nhớ `git pull`).
4. Vẫn không vào được → **mạng công ty có thể chặn `*.gradio.live`** — thử 4G/hotspot, hoặc dùng UI inline.

> Gặp `PDX has already been initialized`: **Runtime → Restart session** rồi chạy lại từ cell 3 (đừng chạy OCR 2 lần trước khi restart).

In [ ]:
from app import build_ui
# ssr_mode=False: gradio 5/6 bật SSR làm share-link ra TRANG TRẮNG trên Colab -> tắt đi
# debug=True: cell chạy mãi là BÌNH THƯỜNG (giữ server sống + hiện log lỗi tại đây)
build_ui().launch(share=True, ssr_mode=False, debug=True)

## 5. (Tuỳ chọn) Benchmark bằng code

In [ ]:
from tcocr.config import PipelineConfig
from tcocr.pipeline import OCRPipeline
from tcocr.benchmark.runner import run_pair

cfg = PipelineConfig(ocr_backend='paddle', escalation_backend='null', correction_backend='protonx')
pipe = OCRPipeline(cfg)
report = run_pair(pipe, 'scan.pdf', 'goc.pdf')   # đổi đường dẫn 2 file của bạn
print(report.to_markdown())

# Đổi ocr_backend='vietocr' rồi chạy lại để so Paddle vs VietOCR.